In [11]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from sklearn.decomposition import FastICA, PCA
import torchvision.transforms as T
import os
from os import listdir
from os.path import join
import seaborn as sns

from mpl_toolkits.axes_grid1 import make_axes_locatable

from tqdm import tqdm

# Tools for the Haar Wavelet decomposition

### Coming from the [HiNet Github repository](https://github.com/TomTomTommi/HiNet)

In [12]:
def to_rgb(image):
    rgb_image = Image.new("RGB", image.size) 
    rgb_image.paste(image)
    return rgb_image 

def dwt_init(x):

    x01 = x[:, :, 0::2, :] / 2
    x02 = x[:, :, 1::2, :] / 2
    x1 = x01[:, :, :, 0::2]
    x2 = x02[:, :, :, 0::2]
    x3 = x01[:, :, :, 1::2]
    x4 = x02[:, :, :, 1::2]
    x_LL = x1 + x2 + x3 + x4
    x_HL = -x1 - x2 + x3 + x4
    x_LH = -x1 + x2 - x3 + x4
    x_HH = x1 - x2 - x3 + x4

    return torch.cat((x_LL, x_HL, x_LH, x_HH), 1)


def iwt_init(x):
    r = 2
    in_batch, in_channel, in_height, in_width = x.size()
    out_batch, out_channel, out_height, out_width = in_batch, int(in_channel / (r ** 2)), r * in_height, r * in_width
    x1 = x[:, 0:out_channel, :, :] / 2
    x2 = x[:, out_channel:out_channel * 2, :, :] / 2
    x3 = x[:, out_channel * 2:out_channel * 3, :, :] / 2
    x4 = x[:, out_channel * 3:out_channel * 4, :, :] / 2

    h = torch.zeros([out_batch, out_channel, out_height, out_width]).float().cuda()

    h[:, :, 0::2, 0::2] = x1 - x2 - x3 + x4
    h[:, :, 1::2, 0::2] = x1 - x2 + x3 - x4
    h[:, :, 0::2, 1::2] = x1 + x2 - x3 - x4
    h[:, :, 1::2, 1::2] = x1 + x2 + x3 + x4

    return h


class DWT(nn.Module):
    def __init__(self):
        super(DWT, self).__init__()
        self.requires_grad = False

    def forward(self, x):
        return dwt_init(x)


class IWT(nn.Module):
    def __init__(self):
        super(IWT, self).__init__()
        self.requires_grad = False

    def forward(self, x):
        return iwt_init(x)
    
dwt = DWT()
iwt = IWT()


In [13]:
def gen_ica(base_path, save_path, pca_sub_ids=[7,8,9,10,11], n_img=10, desc=None):

    list_of_files = sorted(listdir(base_path))

    for im_id in tqdm(range(n_img), desc=desc):
        img = Image.open(join(base_path, list_of_files[im_id]))
        img = np.array(to_rgb(img))

        img = np.swapaxes(img, 0, 2)
        img = np.swapaxes(img, 1, 2)
        img = torch.Tensor(img)
        img = img.unsqueeze(0)
        wavedec = dwt(img)
        wavedec = wavedec.numpy()

        pca = PCA(n_components=12)
        wavedec_pca = pca.fit_transform(wavedec[0].reshape(-1, 256**2).T)

        observations = np.concatenate([
            [wavedec_pca[:,i].flatten() for i in pca_sub_ids]
        ])

        try:
            fastica = FastICA(max_iter=10_000, n_components=2)
            x_transformed = fastica.fit_transform(observations.T)

            np.save(join(save_path, f"{im_id}.npy"), x_transformed)

        except ValueError as e:
            print(im_id, e) 

## Generate the compoents for specific subbands (here 10-8)...

In [14]:
base_path = "/data2/antoine/Hinet/HiNet/image_COCO/steg/"
save_path = f"dataset/hinet/ica_all_pairs/{10}_{8}"
if not os.path.exists(save_path): os.makedirs(save_path)
gen_ica(base_path=base_path, save_path=save_path, pca_sub_ids=[10,8])

base_path = "/data2/antoine/datasets/COCO/COCO_real_512"
save_path = f"dataset/cover/ica_all_pairs/{10}_{8}"
if not os.path.exists(save_path): os.makedirs(save_path)
gen_ica(base_path=base_path, save_path=save_path, pca_sub_ids=[10,8])

100%|██████████| 10/10 [00:01<00:00,  7.06it/s]


## ... Or generate all pairs

In [ ]:
N_IMG = 10 # Change this depending on the number of images you want to manipulate

for i in range(12):
    for j in range(i):
        if i != j:
            base_path = "/data2/antoine/datasets/COCO/COCO_real_512" # Replace to match your cover image folder
            save_path = f"dataset/cover/ica_all_pairs/{i}_{j}/cover"
            if not os.path.exists(save_path): os.makedirs(save_path)
            gen_ica(base_path=base_path, save_path=save_path, pca_sub_ids=[i,j], n_img=N_IMG, desc=f"{i}_{j}/cover")

            base_path = "/data2/antoine/Hinet/HiNet/image_COCO/steg/" # Replace to match your stego image folder
            save_path = f"dataset/hinet/ica_all_pairs/{i}_{j}" # Replace to dataset/<stego_model>/ica_all_pairs/{i}_{j} 
            if not os.path.exists(save_path): os.makedirs(save_path)
            gen_ica(base_path=base_path, save_path=save_path, pca_sub_ids=[i,j], n_img=N_IMG, desc=f"{i}_{j}/stego")